# 10 - Member 3 summary dashboard

**Purpose:** Collect the key numbers from the saved Member 3 runs in one visual dashboard.

**Inputs:** scratch/derived/member3/*.json (rebuilt automatically if missing)

**Outputs:** dashboard figure and summary table

**Status labels:** Each item keeps its own label (VERIFIED / SIMULATOR / BLOCKED / PROVISIONAL).

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
for f, s in (('scratch/derived/member3/scheduling_comparison.json','member3_phase3.py'), ('scratch/derived/member3/verification_evidence.json','member3_phase4.py'), ('scratch/derived/member3/ml_cesnet.json','member3_phase5.py'), ('scratch/derived/member3/bundle/plan.json','member3_pipeline.py'), ('scratch/derived/member3/agent_eval.json','evaluate_agent.py')):
    ensure(f, s)
load = lambda n: json.loads((DERIVED / n).read_text())
plan, sc, ev, ma, mb, ag = (load(n) for n in ('bundle/plan.json','scheduling_comparison.json','verification_evidence.json','ml_cesnet.json','ml_purpose_triage.json','agent_eval.json'))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8.5))
st = pd.Series([s['state'] for s in plan['steps']]).value_counts()
ax[0,0].bar(st.index, st.values, color=['#2a9d8f' if i=='READY' else '#e9c46a' for i in st.index]); ax[0,0].set_title('Migration steps by gate state')
w = pd.Series({k: len(v) for k, v in plan['schedule']['waves'].items()})
ax[0,1].bar([f'Wave {k}' for k in w.index], w.values, color='#457b9d'); ax[0,1].set_title(f"Schedule (objective {plan['schedule']['objective']}, 0 violations)")
q = sc['qaoa_phase2_wave0']; ax[0,2].bar(['QAOA P(opt)', 'uniform'], [q['optimum_probability'], q['uniform_probability']], color=['#457b9d','#9e9e9e']); ax[0,2].set_title('QAOA on simulator (no advantage)')
hb = {k: v['bytes_read'] + v['bytes_written'] for k, v in ev['traces'].items() if isinstance(v, dict) and 'bytes_read' in v}
ax[1,0].bar(['classical', 'hybrid', 'hybrid+ML-DSA cert'], list(hb.values()), color=['#457b9d','#e9c46a','#e76f51']); ax[1,0].set_title('Handshake bytes (loopback)')
mm = {'CESNET model': ma['metrics']['model']['macro_f1'], 'CESNET majority': ma['metrics']['majority_baseline']['macro_f1'], 'triage model': mb['metrics_pooled_over_folds']['model']['macro_f1'], 'triage rules': mb['metrics_pooled_over_folds']['rule_engine']['macro_f1']}
ax[1,1].bar(mm.keys(), mm.values(), color=['#2a9d8f','#9e9e9e','#e76f51','#457b9d']); ax[1,1].set_title('Macro-F1 (ML tracks)'); ax[1,1].tick_params(axis='x', rotation=20); ax[1,1].set_ylim(0, 1)
ax[1,2].pie([ag['passed'], ag['n']-ag['passed']], labels=['passed', 'failed'], colors=['#2a9d8f','#e76f51'], autopct='%1.0f%%'); ax[1,2].set_title(f"Assistant eval {ag['passed']}/{ag['n']}")
for a in (ax[0,0], ax[0,1], ax[0,2], ax[1,0], ax[1,1]):
    for cont in a.containers: a.bar_label(cont, fmt='%g', fontsize=8)
fig.suptitle('CryptoShift Member 3 - results dashboard (synthetic demo bank; loopback test scenario)', fontsize=13)
savefig('10_dashboard')

In [ ]:
rows = [
 ('Plan steps READY', sum(s['state']=='READY' for s in plan['steps']), 'VERIFIED (generic loopback evidence)'),
 ('Baseline objective', sc['baseline']['objective'], 'VERIFIED (synthetic inputs, assumed constraints)'),
 ('QpiAI Opt local solve (objective vs baseline)', f"{sc['qpiai_opt']['authenticated_local_solve']['status']} {sc['qpiai_opt']['authenticated_local_solve'].get('objective', '')} vs {sc['baseline']['objective']}", 'VERIFIED (owner-approved, mode=local; equal, not better)' if sc['qpiai_opt']['authenticated_local_solve']['status'] == 'RUN' else 'BLOCKED'),
 ('QAOA P(optimum) vs uniform', f"{q['optimum_probability']} vs {q['uniform_probability']}", 'SIMULATOR'),
 ('Compatibility cases passed', f"{sum(r['pass'] for r in ev['compatibility_matrix'])}/{len(ev['compatibility_matrix'])}", 'VERIFIED loopback'),
 ('CESNET macro-F1 (model / majority)', f"{mm['CESNET model']} / {mm['CESNET majority']}", 'VERIFIED demo; not crypto'),
 ('Purpose triage beats rules', mb['beats_rule_engine'], 'VERIFIED (tiny sample)'),
 ('Assistant eval passed (template)', f"{ag['passed']}/{ag['n']}", 'VERIFIED (TEMPLATE_FALLBACK)'),
 ('Assistant eval passed (Sarvam cloud, v3 check)', (lambda p: f"{json.loads(p.read_text()).get('v3_passed')}/24" if p.is_file() else 'n/a')(DERIVED / 'agent_eval_sarvam.json'), 'VERIFIED one live run (sarvam-105b)'),
]
pd.DataFrame(rows, columns=['item', 'value', 'status'])